In [52]:
from decimal import Decimal
from datetime import datetime

class Account:
    def __init__(self, account_holder: str, balance: float = 0):
        cents = self._to_cents(balance)
        self._check_initial_balance(cents)
        self.holder = account_holder
        self._balance: int = cents
        self.operations_history: list[dict] = []

    @staticmethod
    def _to_cents(balance: float) -> int:
        """ Домножаем баланс на 100 чтобы избавиться от копеек и избежать IEE754 ошибки"""
        decimal_balance = Decimal(str(balance))
        if not decimal_balance.is_finite():
            raise ValueError(f'Balance must be finite, got {balance!r}')

        return int(decimal_balance * 100)

    def _check_initial_balance(self, cents: int) -> None:
        if cents < 0:
            raise ValueError(
                f'Balance must not be negative, got {self._format_cents(cents)}'
            )

    def _max_withdraw(self) -> int:
        return self._balance

    def _uses_credit(self, cents: int) -> bool:
        return False

    @staticmethod
    def _format_cents(cents: int) -> str:
        sign = '-' if cents < 0 else ''
        dollars, rest = divmod(abs(cents), 100)
        return f'{sign}{dollars}.{rest:02d}'
    
    def add_record_to_operations_history(
        self,
        operation_type: str,
        amount: float,
        status: str,
        used_credit: bool = False,
    ) -> None:
        self.operations_history.append(
            {
                # deposit | withdraw
                'type': operation_type,
                'amount': float(amount),
                'datetime': datetime.now().isoformat(timespec='seconds'),
                'balance_after': self.get_balance(),
                # succsess | fail
                'status': status,
                'used_credit': used_credit,
            }
        )

    def deposit(self, amount: float) -> bool:
        cents = self._to_cents(amount)
        if cents <= 0:
            raise ValueError(f'deposit amount must be positive, got {amount!r}')

        self._balance += cents
        self.add_record_to_operations_history('deposit', amount, 'success')
        return True

    def withdraw(self, amount: float) -> bool:
        cents = self._to_cents(amount)
        if cents <= 0:
            raise ValueError(f'withdraw amount must be positive, got {amount!r}')

        if cents > self._max_withdraw():
            self.add_record_to_operations_history('withdraw', amount, 'fail')
            return False

        used_credit = self._uses_credit(cents)
        self._balance -= cents
        self.add_record_to_operations_history(
            'withdraw', amount, 'success', used_credit=used_credit
        )
        return True
        
    
    def get_balance(self) -> str:
        return self._format_cents(self._balance)

    def get_history(self) -> list[dict]:
        return list(self.operations_history)
   

In [53]:
class CreditAccount(Account):
    def __init__(self, account_holder: str, balance: float = 0, credit_limit: float = 0):
        limit = self._to_cents(credit_limit)
        if limit < 0:
            raise ValueError(f'Credit limit must be non-negtive, got {credit_limit}')

        self._credit_limit: int = limit
        super().__init__(account_holder, balance)

    def _check_initial_balance(self, cents: int) -> None:
        if cents < -self._credit_limit:
            raise ValueError(
                f'balance must be >= -credit_limit'
                f'({self._format_cents(-self._credit_limit)}),'
                f'got {self._format_cents(cents)}'
            )

    def _max_withdraw(self) -> int:
        return self._balance + self._credit_limit

    def _uses_credit(self, cents: int) -> bool:
        return cents > self._balance

    def get_available_credit(self) -> str:
        return self._format_cents(self._balance + self._credit_limit)

In [54]:
"""
Тесты делала ии, на граничные случаи и прочие разумные случаи надеяться
не приходится, но базовый функционал вроде проверяет и все прошло
"""
passed = 0


def check(name, condition, got=None):
    global passed
    if condition:
        passed += 1
        print("PASS -", name)
    else:
        print("FAIL -", name, f"| получено: {got!r}")
        raise AssertionError(name)


def check_raises(name, action):
    try:
        action()
    except ValueError:
        check(name, True)
        return
    check(name, False, "ValueError не возник")


print("=== Этап 1: инициализация ===")
acc = Account("Jack", 200)
check("holder сохранён", acc.holder == "Jack", acc.holder)
check("баланс 200 -> '200.00'", acc.get_balance() == "200.00", acc.get_balance())
check("история пуста", acc.get_history() == [], acc.get_history())
check("дефолтный баланс -> '0.00'", Account("Jack").get_balance() == "0.00")
check_raises("отрицательный старт -> ValueError", lambda: Account("X", -1))
check_raises("nan -> ValueError", lambda: Account("X", float("nan")))
check_raises("inf -> ValueError", lambda: Account("X", float("inf")))

print("\n=== Этап 2: deposit / withdraw ===")
acc.deposit(50.25)
check("deposit: '250.25'", acc.get_balance() == "250.25", acc.get_balance())
check("withdraw(30) вернул True", acc.withdraw(30) is True)
check("withdraw: '220.25'", acc.get_balance() == "220.25", acc.get_balance())
check("нехватка: вернул False", acc.withdraw(1000) is False)
check("нехватка: баланс не изменился", acc.get_balance() == "220.25", acc.get_balance())

op_dep, op_wd, op_fail = acc.get_history()
check("deposit: запись success",
      op_dep["type"] == "deposit" and op_dep["status"] == "success", op_dep)
check("запись из 6 полей (с used_credit)",
      set(op_dep) == {"type", "amount", "datetime", "balance_after", "status", "used_credit"},
      list(op_dep))
check("deposit: used_credit=False", op_dep["used_credit"] is False, op_dep.get("used_credit"))
check("withdraw: запись success",
      op_wd["type"] == "withdraw" and op_wd["status"] == "success", op_wd)
check("нехватка: запись fail", op_fail["status"] == "fail", op_fail)

check_raises("deposit(0) -> ValueError", lambda: acc.deposit(0))
check_raises("deposit(-5) -> ValueError", lambda: acc.deposit(-5))
check_raises("withdraw(-5) -> ValueError", lambda: acc.withdraw(-5))

h = acc.get_history()
h.clear()
check("get_history() вернул копию", len(acc.get_history()) == 3, len(acc.get_history()))

print("\n=== Точность денег ===")
b = Account("X", 200)
b.deposit(0.1)
b.deposit(0.2)
check("0.1 + 0.2 без float-мусора -> '200.30'", b.get_balance() == "200.30", b.get_balance())
check("5 центов -> '0.05'", Account("X", 0.05).get_balance() == "0.05")

print("\n=== Этап 3: CreditAccount ===")
check("CreditAccount — наследник Account", issubclass(CreditAccount, Account))
check("старт -50 при лимите 100 -> '-50.00'",
      CreditAccount("X", -50, credit_limit=100).get_balance() == "-50.00")
check_raises("старт -150 при лимите 100 -> ValueError",
             lambda: CreditAccount("X", -150, credit_limit=100))
check_raises("отрицательный credit_limit -> ValueError",
             lambda: CreditAccount("X", 0, credit_limit=-1))

c = CreditAccount("Jack", 0, credit_limit=100)
check("available = '100.00'", c.get_available_credit() == "100.00", c.get_available_credit())
check("withdraw(30) -> True", c.withdraw(30) is True)
check("баланс '-30.00'", c.get_balance() == "-30.00", c.get_balance())
check("available после снятия = '70.00'", c.get_available_credit() == "70.00", c.get_available_credit())
check("снятие в кредит: used_credit=True", c.get_history()[-1]["used_credit"] is True, c.get_history()[-1])

c2 = CreditAccount("Jack", 100, credit_limit=100)
c2.withdraw(40)
check("снятие из своих средств: used_credit=False", c2.get_history()[-1]["used_credit"] is False, c2.get_history()[-1])

c3 = CreditAccount("Jack", 50, credit_limit=100)
c3.withdraw(100)
check("смешанное 50 своих + 50 кредитных -> '-50.00'", c3.get_balance() == "-50.00", c3.get_balance())
check("смешанное: used_credit=True", c3.get_history()[-1]["used_credit"] is True, c3.get_history()[-1])

c4 = CreditAccount("Jack", 0, credit_limit=100)
c4.withdraw(100)
check("ровно до лимита -> '-100.00'", c4.get_balance() == "-100.00", c4.get_balance())
check("ниже лимита нельзя", c4.withdraw(0.01) is False and c4.get_balance() == "-100.00", c4.get_balance())
check("fail-запись: статус fail, used_credit=False",
      c4.get_history()[-1]["status"] == "fail" and c4.get_history()[-1]["used_credit"] is False,
      c4.get_history()[-1])

c5 = CreditAccount("Jack", 0, credit_limit=100)
c5.withdraw(80)
c5.deposit(30)
check("депозит гасит долг -> '-50.00'", c5.get_balance() == "-50.00", c5.get_balance())
check("available при балансе 200 и лимите 100 -> '300.00'",
      CreditAccount("X", 200, credit_limit=100).get_available_credit() == "300.00")
check("формат минуса -> '-1.50' (а не '-2.50')",
      CreditAccount("X", -1.5, credit_limit=100).get_balance() == "-1.50")

a1, a2 = Account("A", 10), Account("B", 10)
a1.deposit(5)
check("счета изолированы",
      a2.get_balance() == "10.00" and a2.get_history() == [],
      (a2.get_balance(), a2.get_history()))

print(f"\nИТОГО: {passed} проверок пройдено")


=== Этап 1: инициализация ===
PASS - holder сохранён
PASS - баланс 200 -> '200.00'
PASS - история пуста
PASS - дефолтный баланс -> '0.00'
PASS - отрицательный старт -> ValueError
PASS - nan -> ValueError
PASS - inf -> ValueError

=== Этап 2: deposit / withdraw ===
PASS - deposit: '250.25'
PASS - withdraw(30) вернул True
PASS - withdraw: '220.25'
PASS - нехватка: вернул False
PASS - нехватка: баланс не изменился
PASS - deposit: запись success
PASS - запись из 6 полей (с used_credit)
PASS - deposit: used_credit=False
PASS - withdraw: запись success
PASS - нехватка: запись fail
PASS - deposit(0) -> ValueError
PASS - deposit(-5) -> ValueError
PASS - withdraw(-5) -> ValueError
PASS - get_history() вернул копию

=== Точность денег ===
PASS - 0.1 + 0.2 без float-мусора -> '200.30'
PASS - 5 центов -> '0.05'

=== Этап 3: CreditAccount ===
PASS - CreditAccount — наследник Account
PASS - старт -50 при лимите 100 -> '-50.00'
PASS - старт -150 при лимите 100 -> ValueError
PASS - отрицательный credi